# Lab 1 — Khởi động Xử lý ngôn ngữ tự nhiên

**Chủ đề:** Dữ liệu văn bản, xử lý chuỗi, biểu thức chính quy, làm sạch văn bản và tokenization cơ bản.  
**Học phần:** Xử lý ngôn ngữ tự nhiên  
**Giảng viên:** Phạm Tiến Lâm

## Mục tiêu thực hành

Sau bài thực hành này, sinh viên có thể:

1. Làm việc với văn bản như một dạng dữ liệu tính toán.
2. Đọc, ghi và thống kê tệp văn bản thuần (`.txt`).
3. Sử dụng biểu thức chính quy để trích xuất thông tin có mẫu.
4. Làm sạch và chuẩn hóa văn bản ở mức cơ bản.
5. Tách token và tính tần suất từ.
6. Thử đọc nội dung từ PDF và HTML ở mức khởi động.

## 0. Chuẩn bị môi trường

Notebook này ưu tiên sử dụng các thư viện chuẩn của Python để có thể chạy độc lập.  
Các phần PDF và HTML có dùng thư viện tùy chọn (`PyPDF2`, `bs4`). Nếu thiếu thư viện, sinh viên có thể cài thêm khi chạy trên Google Colab.

In [2]:
import re
import json
import string
from collections import Counter
from pathlib import Path

print("Python environment is ready.")

Python environment is ready.


## 1. Văn bản như dữ liệu

Trong NLP, văn bản không chỉ là nội dung để con người đọc, mà còn là **dữ liệu đầu vào** cho các thuật toán.

Một pipeline tối giản thường gồm:

```text
Raw text → Cleaning → Tokenization → Representation → Model → Evaluation
```

In [1]:
text_vi = """
Xử lý ngôn ngữ tự nhiên là một lĩnh vực của trí tuệ nhân tạo.
NLP giúp máy tính phân tích, hiểu và sinh ngôn ngữ tự nhiên.
Ví dụ: tìm kiếm thông tin, phân loại văn bản, dịch máy, tóm tắt và chatbot.
"""

text_en = "Natural Language Processing helps computers process human language."

print(text_vi)
print("Số ký tự:", len(text_vi))
print("Số dòng:", len(text_vi.strip().splitlines()))


Xử lý ngôn ngữ tự nhiên là một lĩnh vực của trí tuệ nhân tạo.
NLP giúp máy tính phân tích, hiểu và sinh ngôn ngữ tự nhiên.
Ví dụ: tìm kiếm thông tin, phân loại văn bản, dịch máy, tóm tắt và chatbot.

Số ký tự: 200
Số dòng: 3


### Câu hỏi nhanh

1. Trong biến `text_vi`, đâu là dữ liệu thô?  
2. Nếu dùng trực tiếp văn bản này cho mô hình học máy, ta sẽ gặp những vấn đề gì?  
3. Những bước tiền xử lý nào có thể cần thiết?

## 2. Làm việc với chuỗi trong Python

Một số thao tác cơ bản:

- `lower()`: chuyển chữ thường
- `replace()`: thay thế chuỗi con
- `split()`: tách chuỗi theo khoảng trắng hoặc ký tự cho trước
- `strip()`: bỏ khoảng trắng ở đầu/cuối

In [3]:
s = "  NLP, Machine Learning, and Text Mining!  "

print("Original:", repr(s))
print("Strip:", repr(s.strip()))
print("Lower:", s.lower())
print("Replace:", s.replace("Text Mining", "Information Extraction"))
print("Split by comma:", [x.strip() for x in s.split(",")])

Original: '  NLP, Machine Learning, and Text Mining!  '
Strip: 'NLP, Machine Learning, and Text Mining!'
Lower:   nlp, machine learning, and text mining!  
Replace:   NLP, Machine Learning, and Information Extraction!  
Split by comma: ['NLP', 'Machine Learning', 'and Text Mining!']


### Bài tập 1

Viết code để:

1. Chuyển `text_vi` thành chữ thường.
2. Thay cụm `trí tuệ nhân tạo` bằng `AI`.
3. Đếm số lần xuất hiện của từ `ngôn ngữ`.

In [4]:
# TODO: Bài tập 1

lower_text = text_vi.lower()
replaced_text = lower_text.replace("trí tuệ nhân tạo", "AI")
count_phrase = lower_text.count("ngôn ngữ")

print(replaced_text)
print("Số lần xuất hiện của 'ngôn ngữ':", count_phrase)


xử lý ngôn ngữ tự nhiên là một lĩnh vực của AI.
nlp giúp máy tính phân tích, hiểu và sinh ngôn ngữ tự nhiên.
ví dụ: tìm kiếm thông tin, phân loại văn bản, dịch máy, tóm tắt và chatbot.

Số lần xuất hiện của 'ngôn ngữ': 2


## 3. Đọc và ghi tệp văn bản thuần

Trong các bài toán NLP, dữ liệu có thể đến từ nhiều nguồn. Định dạng đơn giản nhất là tệp `.txt`.

In [5]:
# Tạo một tệp văn bản mẫu
sample_path = Path("sample_nlp_text.txt")

with open(sample_path, "w", encoding="utf-8") as f:
    f.write("Dòng 1: NLP xử lý ngôn ngữ tự nhiên.\n")
    f.write("Dòng 2: Văn bản là dữ liệu quan trọng.\n")
    f.write("Dòng 3: Python hỗ trợ xử lý văn bản hiệu quả.\n")

# Ghi thêm nội dung vào cuối tệp
with open(sample_path, "a", encoding="utf-8") as f:
    f.write("Dòng 4: Biểu thức chính quy giúp tìm mẫu trong văn bản.\n")

# Đọc lại toàn bộ tệp
with open(sample_path, "r", encoding="utf-8") as f:
    content = f.read()

print(content)

Dòng 1: NLP xử lý ngôn ngữ tự nhiên.
Dòng 2: Văn bản là dữ liệu quan trọng.
Dòng 3: Python hỗ trợ xử lý văn bản hiệu quả.
Dòng 4: Biểu thức chính quy giúp tìm mẫu trong văn bản.



In [6]:
# Thống kê cơ bản
lines = content.strip().splitlines()
words = content.split()
characters = len(content)

print("Số dòng:", len(lines))
print("Số từ thô:", len(words))
print("Số ký tự:", characters)

Số dòng: 4
Số từ thô: 41
Số ký tự: 178


### Bài tập 2

Tạo một tệp `student_intro.txt` gồm 3–5 dòng giới thiệu bản thân hoặc chủ đề quan tâm.  
Sau đó đọc tệp và tính:

1. Số dòng
2. Số từ
3. Số ký tự
4. Từ dài nhất trong văn bản

In [7]:
# TODO: Bài tập 2

student_path = Path("student_intro.txt")

with open(student_path, "w", encoding="utf-8") as f:
    f.write("Tôi đang học xử lý ngôn ngữ tự nhiên.\n")
    f.write("Tôi muốn xây dựng một hệ thống phân tích văn bản tiếng Việt.\n")
    f.write("Python là công cụ thực hành chính trong học phần này.\n")

student_text = student_path.read_text(encoding="utf-8")
student_words = student_text.split()

print(student_text)
print("Số dòng:", len(student_text.strip().splitlines()))
print("Số từ:", len(student_words))
print("Số ký tự:", len(student_text))
print("Từ dài nhất:", max(student_words, key=len))

Tôi đang học xử lý ngôn ngữ tự nhiên.
Tôi muốn xây dựng một hệ thống phân tích văn bản tiếng Việt.
Python là công cụ thực hành chính trong học phần này.

Số dòng: 3
Số từ: 33
Số ký tự: 153
Từ dài nhất: nhiên.


## 4. Biểu thức chính quy — Regular Expression

Biểu thức chính quy là một ngôn ngữ để mô tả **mẫu chuỗi ký tự**.  
Nó thường dùng để trích xuất email, số điện thoại, ngày tháng, URL, mã định danh hoặc loại bỏ nhiễu trong văn bản.

In [8]:
contact_text = """
Liên hệ Khoa Khoa học máy tính qua email: cs@phenikaa-uni.edu.vn.
Hotline: 024-1234-5678 hoặc 0987 654 321.
Website: https://phenikaa-uni.edu.vn
Ngày cập nhật: 25/09/2026.
"""

print(contact_text)


Liên hệ Khoa Khoa học máy tính qua email: cs@phenikaa-uni.edu.vn.
Hotline: 024-1234-5678 hoặc 0987 654 321.
Website: https://phenikaa-uni.edu.vn
Ngày cập nhật: 25/09/2026.



In [9]:
# Một số mẫu regex cơ bản
email_pattern = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
phone_pattern = r"(?:\d{3,4}[-\s]?\d{3,4}[-\s]?\d{3,4})"
url_pattern = r"https?://\S+"
date_pattern = r"\d{1,2}/\d{1,2}/\d{4}"

emails = re.findall(email_pattern, contact_text)
phones = re.findall(phone_pattern, contact_text)
urls = re.findall(url_pattern, contact_text)
dates = re.findall(date_pattern, contact_text)

print("Emails:", emails)
print("Phones:", phones)
print("URLs:", urls)
print("Dates:", dates)

Emails: ['cs@phenikaa-uni.edu.vn']
Phones: ['024-1234-5678', '0987 654 321']
URLs: ['https://phenikaa-uni.edu.vn']
Dates: ['25/09/2026']


### Một số ký hiệu thường gặp trong regex

| Ký hiệu | Ý nghĩa ngắn gọn |
|---|---|
| `.` | Bất kỳ ký tự nào, trừ xuống dòng |
| `*` | Lặp 0 hoặc nhiều lần |
| `+` | Lặp 1 hoặc nhiều lần |
| `?` | Có hoặc không có |
| `[abc]` | Một trong các ký tự a, b, c |
| `\d` | Chữ số |
| `\s` | Khoảng trắng |
| `^` | Bắt đầu chuỗi |
| `$` | Kết thúc chuỗi |
| `` | Ranh giới từ |

### Bài tập 3

Từ đoạn văn bản `contact_text`, hãy viết regex để trích xuất:

1. Tất cả các số có trong văn bản.
2. Tên miền website, ví dụ: `phenikaa-uni.edu.vn`.
3. Các chuỗi có chứa dấu gạch nối `-`.

In [10]:
# TODO: Bài tập 3

all_numbers = re.findall(r"\d+", contact_text)
domains = re.findall(r"https?://([^\s/]+)", contact_text)
hyphen_words = re.findall(r"\b[\w.-]*-[\w.-]*\b", contact_text)

print("All numbers:", all_numbers)
print("Domains:", domains)
print("Hyphenated strings:", hyphen_words)

All numbers: ['024', '1234', '5678', '0987', '654', '321', '25', '09', '2026']
Domains: ['phenikaa-uni.edu.vn']
Hyphenated strings: ['phenikaa-uni.edu.vn', '024-1234-5678', 'phenikaa-uni.edu.vn']


## 5. Làm sạch và chuẩn hóa văn bản

Mục tiêu của làm sạch văn bản là giảm nhiễu nhưng không làm mất thông tin quan trọng.  
Các thao tác thường gặp:

- Chuyển chữ thường
- Loại bỏ URL
- Loại bỏ hoặc chuẩn hóa dấu câu
- Chuẩn hóa khoảng trắng
- Loại bỏ stopwords khi phù hợp

In [11]:
noisy_text = """
🔥 NLP!!! là    một lĩnh vực rất thú vị... Xem thêm tại https://example.com/nlp?id=123.
Email liên hệ: student@example.com    #AI #NLP
"""

print(noisy_text)


🔥 NLP!!! là    một lĩnh vực rất thú vị... Xem thêm tại https://example.com/nlp?id=123.
Email liên hệ: student@example.com    #AI #NLP



In [12]:
def clean_text_basic(text: str) -> str:
    """Làm sạch văn bản cơ bản, phù hợp cho ví dụ nhập môn."""
    text = text.lower()
    text = re.sub(r"https?://\S+", " ", text)       # bỏ URL
    text = re.sub(r"[\w.+-]+@[\w-]+\.[\w.-]+", " ", text)  # bỏ email
    text = re.sub(r"[^\w\sàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ]", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

cleaned = clean_text_basic(noisy_text)
print(cleaned)

nlp là một lĩnh vực rất thú vị xem thêm tại email liên hệ ai nlp


### Câu hỏi thảo luận

1. Hàm `clean_text_basic` đã loại bỏ những thông tin nào?  
2. Trong bài toán phân loại spam, có nên xóa email và URL không? Vì sao?  
3. Trong bài toán trích xuất thông tin liên hệ, có nên xóa email và số điện thoại không?

## 6. Tokenization cơ bản

Tokenization là quá trình tách văn bản thành các đơn vị xử lý nhỏ hơn.  
Với tiếng Anh, tokenization theo khoảng trắng có thể tương đối ổn trong ví dụ đơn giản.  
Với tiếng Việt, phân đoạn từ phức tạp hơn vì một từ có thể gồm nhiều âm tiết, ví dụ: `xử lý`, `ngôn ngữ`, `tự nhiên`.

In [13]:
def simple_tokenize(text: str):
    """Tokenizer đơn giản dựa trên regex, dùng cho minh họa nhập môn."""
    return re.findall(r"\b\w+\b", text.lower(), flags=re.UNICODE)

tokens = simple_tokenize(cleaned)
print(tokens)
print("Số token:", len(tokens))

['nlp', 'là', 'một', 'lĩnh', 'vực', 'rất', 'thú', 'vị', 'xem', 'thêm', 'tại', 'email', 'liên', 'hệ', 'ai', 'nlp']
Số token: 16


In [14]:
# Stopwords tối giản để minh họa
stopwords_vi = {"là", "một", "và", "của", "cho", "trong", "với", "tại", "rất"}

filtered_tokens = [tok for tok in tokens if tok not in stopwords_vi]
print(filtered_tokens)

['nlp', 'lĩnh', 'vực', 'thú', 'vị', 'xem', 'thêm', 'email', 'liên', 'hệ', 'ai', 'nlp']


In [15]:
# Tần suất token
freq = Counter(filtered_tokens)
print(freq.most_common(10))

[('nlp', 2), ('lĩnh', 1), ('vực', 1), ('thú', 1), ('vị', 1), ('xem', 1), ('thêm', 1), ('email', 1), ('liên', 1), ('hệ', 1)]


### Bài tập 4

Viết một hàm `preprocess(text)` thực hiện các bước:

1. Làm sạch văn bản bằng `clean_text_basic`.
2. Tách token bằng `simple_tokenize`.
3. Loại bỏ stopwords tiếng Việt.
4. Trả về danh sách token sau xử lý.

In [16]:
# TODO: Bài tập 4

def preprocess(text: str):
    cleaned = clean_text_basic(text)
    tokens = simple_tokenize(cleaned)
    tokens = [tok for tok in tokens if tok not in stopwords_vi]
    return tokens

sample = "NLP là một lĩnh vực của AI. NLP xử lý văn bản, tiếng nói và hội thoại."
print(preprocess(sample))

['nlp', 'lĩnh', 'vực', 'ai', 'nlp', 'xử', 'lý', 'văn', 'bản', 'tiếng', 'nói', 'hội', 'thoại']


## 7. Mini corpus và thống kê từ vựng

Một corpus là một tập văn bản dùng để phân tích hoặc huấn luyện mô hình.  
Trong ví dụ nhỏ này, mỗi phần tử của danh sách là một tài liệu.

In [17]:
corpus = [
    "NLP giúp máy tính xử lý ngôn ngữ tự nhiên.",
    "Phân loại văn bản là một bài toán quan trọng trong NLP.",
    "Biểu thức chính quy giúp trích xuất email, số điện thoại và URL.",
    "Tokenization là bước tiền xử lý quan trọng cho mô hình ngôn ngữ.",
    "Chatbot và hệ hỏi đáp là các ứng dụng phổ biến của NLP."
]

processed_corpus = [preprocess(doc) for doc in corpus]
for i, toks in enumerate(processed_corpus, 1):
    print(f"Document {i}:", toks)

Document 1: ['nlp', 'giúp', 'máy', 'tính', 'xử', 'lý', 'ngôn', 'ngữ', 'tự', 'nhiên']
Document 2: ['phân', 'loại', 'văn', 'bản', 'bài', 'toán', 'quan', 'trọng', 'nlp']
Document 3: ['biểu', 'thức', 'chính', 'quy', 'giúp', 'trích', 'xuất', 'email', 'số', 'điện', 'thoại', 'url']
Document 4: ['tokenization', 'bước', 'tiền', 'xử', 'lý', 'quan', 'trọng', 'mô', 'hình', 'ngôn', 'ngữ']
Document 5: ['chatbot', 'hệ', 'hỏi', 'đáp', 'các', 'ứng', 'dụng', 'phổ', 'biến', 'nlp']


In [18]:
# Xây dựng từ vựng
vocab = sorted(set(token for doc in processed_corpus for token in doc))
print("Vocabulary size:", len(vocab))
print(vocab)

Vocabulary size: 43
['biến', 'biểu', 'bài', 'bước', 'bản', 'chatbot', 'chính', 'các', 'dụng', 'email', 'giúp', 'hình', 'hệ', 'hỏi', 'loại', 'lý', 'máy', 'mô', 'ngôn', 'ngữ', 'nhiên', 'nlp', 'phân', 'phổ', 'quan', 'quy', 'số', 'thoại', 'thức', 'tiền', 'tokenization', 'toán', 'trích', 'trọng', 'tính', 'tự', 'url', 'văn', 'xuất', 'xử', 'điện', 'đáp', 'ứng']


In [19]:
# Đếm tần suất toàn corpus
corpus_freq = Counter(token for doc in processed_corpus for token in doc)
corpus_freq.most_common(10)

[('nlp', 3),
 ('giúp', 2),
 ('xử', 2),
 ('lý', 2),
 ('ngôn', 2),
 ('ngữ', 2),
 ('quan', 2),
 ('trọng', 2),
 ('máy', 1),
 ('tính', 1)]

## 8. Đọc HTML ở mức cơ bản

Trong thực tế, nhiều dữ liệu văn bản đến từ trang web.  
Ở đây ta dùng một đoạn HTML mẫu, không cần truy cập Internet.

In [20]:
html_doc = """
<html>
  <head><title>Giới thiệu NLP</title></head>
  <body>
    <h1>Xử lý ngôn ngữ tự nhiên</h1>
    <p>NLP là lĩnh vực nghiên cứu cách máy tính xử lý ngôn ngữ của con người.</p>
    <p>Các ứng dụng gồm tìm kiếm, phân loại văn bản, dịch máy và chatbot.</p>
    <a href="https://phenikaa-uni.edu.vn">Phenikaa University</a>
  </body>
</html>
"""

try:
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(html_doc, "html.parser")
    print("Title:", soup.title.get_text())
    print("Paragraphs:")
    for p in soup.find_all("p"):
        print("-", p.get_text())
    print("Links:", [a.get("href") for a in soup.find_all("a")])
except ImportError:
    print("Chưa có bs4. Có thể cài bằng: !pip install beautifulsoup4")

Title: Giới thiệu NLP
Paragraphs:
- NLP là lĩnh vực nghiên cứu cách máy tính xử lý ngôn ngữ của con người.
- Các ứng dụng gồm tìm kiếm, phân loại văn bản, dịch máy và chatbot.
Links: ['https://phenikaa-uni.edu.vn']


## 9. Đọc PDF ở mức cơ bản

Phần này là tùy chọn. Khi chạy trên Google Colab, sinh viên có thể upload một tệp PDF và dùng `PyPDF2` để trích xuất văn bản.

> Lưu ý: PDF có thể chứa layout phức tạp hoặc ảnh scan, nên trích xuất văn bản có thể không hoàn hảo.

In [22]:
# Optional: đọc PDF nếu có file PDF trong thư mục làm việc
# Bước 1: cài PyPDF2 nếu cần: !pip install PyPDF2
# Bước 2: đặt đường dẫn file PDF vào biến pdf_path

pdf_path = Path("sample.pdf")

if pdf_path.exists():
    try:
        import PyPDF2
        with open(pdf_path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            print("Số trang:", len(reader.pages))
            first_page_text = reader.pages[0].extract_text()
            print(first_page_text[:1000])
    except ImportError:
        print("Chưa có PyPDF2. Cài bằng: !pip install PyPDF2")
else:
    print("Không tìm thấy sample.pdf. Hãy upload một file PDF và đổi tên/đường dẫn cho phù hợp.")

Không tìm thấy sample.pdf. Hãy upload một file PDF và đổi tên/đường dẫn cho phù hợp.


## 10. Mini challenge cuối buổi

Cho đoạn văn bản dưới đây, hãy xây dựng một pipeline xử lý gồm:

1. Trích xuất email, số điện thoại và URL.
2. Làm sạch văn bản.
3. Tokenization.
4. Loại bỏ stopwords.
5. In ra 10 token phổ biến nhất.

In [23]:
challenge_text = """
Khoa Khoa học máy tính thông báo mở lớp NLP vào ngày 01/10/2026.
Sinh viên đăng ký qua email nlp-course@phenikaa-uni.edu.vn hoặc hotline 024 9999 8888.
Thông tin chi tiết xem tại https://phenikaa-uni.edu.vn/nlp-course.
NLP là học phần kết hợp giữa ngôn ngữ học, thống kê, học máy và trí tuệ nhân tạo.
"""

print(challenge_text)


Khoa Khoa học máy tính thông báo mở lớp NLP vào ngày 01/10/2026.
Sinh viên đăng ký qua email nlp-course@phenikaa-uni.edu.vn hoặc hotline 024 9999 8888.
Thông tin chi tiết xem tại https://phenikaa-uni.edu.vn/nlp-course.
NLP là học phần kết hợp giữa ngôn ngữ học, thống kê, học máy và trí tuệ nhân tạo.



In [24]:
# TODO: Mini challenge

emails = re.findall(email_pattern, challenge_text)
phones = re.findall(phone_pattern, challenge_text)
urls = re.findall(url_pattern, challenge_text)
challenge_tokens = preprocess(challenge_text)
challenge_freq = Counter(challenge_tokens)

print("Emails:", emails)
print("Phones:", phones)
print("URLs:", urls)
print("Tokens:", challenge_tokens)
print("Top tokens:", challenge_freq.most_common(10))

Emails: ['nlp-course@phenikaa-uni.edu.vn']
Phones: ['024 9999 8888']
URLs: ['https://phenikaa-uni.edu.vn/nlp-course.']
Tokens: ['khoa', 'khoa', 'học', 'máy', 'tính', 'thông', 'báo', 'mở', 'lớp', 'nlp', 'vào', 'ngày', '01', '10', '2026', 'sinh', 'viên', 'đăng', 'ký', 'qua', 'email', 'hoặc', 'hotline', '024', '9999', '8888', 'thông', 'tin', 'chi', 'tiết', 'xem', 'nlp', 'học', 'phần', 'kết', 'hợp', 'giữa', 'ngôn', 'ngữ', 'học', 'thống', 'kê', 'học', 'máy', 'trí', 'tuệ', 'nhân', 'tạo']
Top tokens: [('học', 4), ('khoa', 2), ('máy', 2), ('thông', 2), ('nlp', 2), ('tính', 1), ('báo', 1), ('mở', 1), ('lớp', 1), ('vào', 1)]


## 11. Câu hỏi phản tư

Trả lời ngắn gọn vào cuối notebook:

1. Tại sao tiền xử lý văn bản là bước quan trọng trong NLP?
2. Khi nào nên dùng regex? Khi nào không nên chỉ dựa vào regex?
3. Vì sao tokenization tiếng Việt khó hơn tokenization tiếng Anh trong nhiều trường hợp?
4. Một quyết định tiền xử lý sai có thể ảnh hưởng thế nào đến mô hình phía sau?

In [25]:
# TODO: Viết câu trả lời phản tư tại đây dưới dạng chuỗi nhiều dòng.
reflection = """
1. ...
2. ...
3. ...
4. ...
"""
print(reflection)


1. ...
2. ...
3. ...
4. ...



## 12. Nộp bài

Sinh viên nộp:

1. File notebook `.ipynb` đã chạy đầy đủ.
2. Kết quả các bài tập TODO.
3. Câu trả lời phản tư cuối bài.

Khuyến nghị: ghi rõ phiên bản thư viện nếu sử dụng thư viện ngoài.